In [1]:
"""
Two Quotients of One Machine — unified runnable system
Dean Kulik / QuHarmonics — v1.0 + v5 + congruence/locality patch

Self-contained: two independent engines sharing no code,
plus Phi (run-signature) machine with bounded local update,
plus arena homology, witness, normal form, staircase, symmetry.

Run: python two_quotients_system.py
Requires: numpy
"""

import itertools, math, random, time
from functools import lru_cache
import numpy as np

P = 1000003  # prime for homology (char 0 behaviour since |G| = 2^n)

# ============================================================
# READER: brute force Myhill-Nerode
# ============================================================
def brute_profile(n, order):
    """Myhill-Nerode state count after each prefix length."""
    pairs = [(0,0),(0,1),(1,0),(1,1)]
    prof = []
    for t in range(n+1):
        known, unknown = order[:t], order[t:]
        sigs = set()
        for kv in itertools.product(pairs, repeat=t):
            outs = []
            for uv in itertools.product(pairs, repeat=n-t):
                a = b = 0
                for p,(x,y) in zip(known,kv):   a |= x<<p; b |= y<<p
                for p,(x,y) in zip(unknown,uv): a |= x<<p; b |= y<<p
                s = (a+b) % (1<<n)
                outs.append(tuple((s>>i)&1 for i in range(n)))
            pending = tuple(i for i in range(n) if len({o[i] for o in outs}) > 1)
            sigs.add((pending, tuple(tuple(o[i] for i in pending) for o in outs)))
        prof.append(len(sigs))
    return prof

def runs_of(S, n):
    out = []
    for p in sorted(S):
        if out and out[-1][1] == p-1: out[-1][1] = p
        else: out.append([p,p])
    return [tuple(r) for r in out]

def F(S, n):
    v = 1
    for lo,hi in runs_of(S,n):
        L = hi-lo+1; bot, top = lo==0, hi==n-1
        v *= 1 if (bot and top) else 2 if bot else (L+1) if top else (2*L+1)
    return v

def pred_profile(n, order): return [F(set(order[:t]), n) for t in range(n+1)]

def named_orders(n):
    mid = (n-1)//2; mo, up, dn, turn = [mid], mid+1, mid-1, True
    while len(mo) < n:
        if turn and up <= n-1: mo.append(up); up+=1
        elif dn >=0: mo.append(dn); dn-=1
        elif up <= n-1: mo.append(up); up+=1
        turn = not turn
    return {"low->high": list(range(n)), "high->low": list(range(n-1,-1,-1)),
            "even/odd": [i for i in range(n) if i%2==0]+[i for i in range(n) if i%2],
            "middle-out": mo}

# ============================================================
# READER: Phi = run signatures (first nontransparent + last const)
# This is the congruence that induces Myhill-Nerode.
# Run transfer: (first_idx or None, last_const or None)
# first_idx = index of first kill/set in run (0..L-1) or None if all transparent
# last_const = const of last kill/set (0/1) or None if all transparent
# ============================================================
def run_transfer_from_assign(run, assign):
    """assign: dict q -> (a_q,b_q). run: (l,r). Returns (f, lc)."""
    l,r = run
    first = None
    last = None
    for idx,q in enumerate(range(l, r+1)):
        a,b = assign[q]
        is_trans = (a ^ b) == 1
        if not is_trans:  # kill or set
            if first is None:
                first = idx
            last = a & b
    return first, last

def phi_of_set(S, assign, n):
    """Phi(S) = tuple of ((l,r), (f,lc), run_type). assign covers S."""
    out = []
    for run in runs_of(S,n):
        l,r = run
        L = r-l+1
        bot, top = l==0, r==n-1
        f,lc = run_transfer_from_assign(run, assign)
        if bot and top:
            sig = ()  # full run 1 state
        elif bot:
            # bottom-anchored: only final carry matters, 0 if all transparent
            sig = (lc if lc is not None else 0,)  # 0 or 1
        elif top:
            # top: only first matters, L+1 possibilities
            sig = (f,)  # None means all transparent
        else:
            # interior: (f, lc) where f=None => transparent
            sig = (f, lc)
        out.append((run, sig))
    return tuple(out)

def phi_equal(phi1, phi2):
    return phi1 == phi2

# Local update U: merge left + middle single + right using only transfers
def merge_transfers(Tl, Tm, Tr, Ll, Lr):
    """
    Tl = (f_l, lc_l), Tm = (f_m, lc_m) for single pos, Tr = (f_r, lc_r)
    Ll = len left, Lr = len right. Returns (f, lc) for combined run length Ll+1+Lr
    """
    fl, lcl = Tl
    fm, lcm = Tm
    fr, lcr = Tr
    # first
    if fl is not None:
        f = fl
    elif fm is not None:
        f = Ll
    elif fr is not None:
        f = Ll+1+fr
    else:
        f = None
    # last const
    if lcr is not None:
        lc = lcr
    elif lcm is not None:
        lc = lcm
    elif lcl is not None:
        lc = lcl
    else:
        lc = None
    return f, lc

def local_update(S, Phi, p, ab, n):
    """
    S: set of read positions before update
    Phi: result of phi_of_set(S, assign, n) for some assign
    p: new position not in S, ab=(a_p,b_p)
    Returns (S', Phi') computed ONLY from S, Phi, p, ab (no assign)
    """
    assert p not in S
    Sp = set(S); Sp.add(p)
    # build map run -> (f,lc) from Phi for quick lookup
    run_to_transfer = {}
    for run,sig in Phi:
        l,r = run
        bot, top = l==0, r==n-1
        if bot and top:
            run_to_transfer[run] = (None, None)  # full run special, but won't be extended unless n=1
        elif bot:
            # sig = (final_carry,)
            final = sig[0]
            # need to recover f,lc? For bottom-anchored, we lost f, but we have last const = final if run had kill/set else None with final 0 meaning all transparent.
            # For update we need f and lc. For bottom-anchored all-transparent, final 0 could mean all transparent -> lc=None. For final 1, must have kill/set -> lc=1 but f unknown.
            # However for bottom-anchored, f is needed only to compute new first when merging upward.
            # We can store f in Phi for bottom-anchored as well to make update local.
            # To keep simple, we will store full (f,lc) even for bottom-anchored in Phi_extended, but for this demo we reconstruct approximate:
            # If final==0, could be all transparent (f=None,lc=None) or last const 0. Both give same final but different f.
            # So we need richer Phi. For true local machine we store (f,lc) for all runs.
            # Here we assume Phi stores (f,lc) for all.
            # Workaround: if we only have final, we cannot distinguish. So we will use extended Phi below.
            pass
        else:
            pass
    # For clean implementation we recompute using extended representation:
    # We will use function phi_extended that stores (f,lc) for every run regardless of type.
    raise NotImplementedError("Use phi_extended + local_update_extended below for true locality")

def phi_extended_of_set(S, assign, n):
    """Stores (f,lc) for every run, regardless of type — needed for local update."""
    out = []
    for run in runs_of(S,n):
        f,lc = run_transfer_from_assign(run, assign)
        out.append((run, (f,lc)))
    return tuple(out)

def local_update_extended(S, Phi_ext, p, ab, n):
    """
    True local update using only S geometry + Phi_ext + new pair.
    Phi_ext = tuple of (run, (f,lc)) as from phi_extended_of_set
    ab = (a_p,b_p)
    Returns (Sp, Phi_ext')
    """
    Sp = set(S); Sp.add(p)
    # map run -> (f,lc)
    mp = {run: trans for run,trans in Phi_ext}
    runs = runs_of(S,n)
    # find neighbours of p
    left_run = None
    right_run = None
    for run in runs:
        l,r = run
        if r == p-1: left_run = run
        if l == p+1: right_run = run

    # transfer of middle single
    a,b = ab
    is_trans = (a ^ b)==1
    if is_trans:
        Tm = (None, None)
    else:
        Tm = (0, a & b)  # f=0, lc=const

    if left_run and right_run:
        # merge left + middle + right
        Tl = mp[left_run]
        Tr = mp[right_run]
        Ll = left_run[1]-left_run[0]+1
        Lr = right_run[1]-right_run[0]+1
        f,lc = merge_transfers(Tl, Tm, Tr, Ll, Lr)
        new_run = (left_run[0], right_run[1])
        # build new Phi_ext: remove left/right, add new_run
        new_phi = []
        for run,trans in Phi_ext:
            if run==left_run or run==right_run: continue
            new_phi.append((run, trans))
        new_phi.append((new_run, (f,lc)))
        new_phi = tuple(sorted(new_phi))
    elif left_run:
        Tl = mp[left_run]
        Ll = left_run[1]-left_run[0]+1
        Tr_empty = (None,None)
        f,lc = merge_transfers(Tl, Tm, Tr_empty, Ll, 0)
        new_run = (left_run[0], p)
        new_phi = [(run,trans) for run,trans in Phi_ext if run!=left_run]
        new_phi.append((new_run,(f,lc)))
        new_phi = tuple(sorted(new_phi))
    elif right_run:
        Tr = mp[right_run]
        Lr = right_run[1]-right_run[0]+1
        Tl_empty = (None,None)
        f,lc = merge_transfers(Tl_empty, Tm, Tr, 0, Lr)
        new_run = (p, right_run[1])
        new_phi = [(run,trans) for run,trans in Phi_ext if run!=right_run]
        new_phi.append((new_run,(f,lc)))
        new_phi = tuple(sorted(new_phi))
    else:
        # isolated new singleton run
        new_run = (p,p)
        f,lc = Tm
        new_phi = list(Phi_ext) + [(new_run,(f,lc))]
        new_phi = tuple(sorted(new_phi))

    # sort by run start for determinism
    new_phi = tuple(sorted(new_phi, key=lambda x: x[0][0]))
    return Sp, new_phi

def count_assignments_for_S(S,n):
    """Iterate over all 4^|S| assignments."""
    positions = sorted(S)
    for prod in itertools.product([(0,0),(0,1),(1,0),(1,1)], repeat=len(positions)):
        assign = dict(zip(positions, prod))
        yield assign

# ============================================================
# ARENA: Cayley complex
# ============================================================
def arena(n):
    N = 1<<n; eidx, elist = {}, []
    for x in range(N):
        for i in range(n):
            y = (x + (1<<i)) % N
            key = (min(x,y), max(x,y))
            if key not in eidx:
                eidx[key]=len(elist); elist.append((x,i,y))
    return N, eidx, elist

def sgn_edge(eidx, x, i, n):
    y = (x + (1<<i)) % (1<<n); k = eidx[(min(x,y), max(x,y))]
    return k, (1 if x < y else -1)

def disjoint(x,i,j,n):
    N=1<<n
    return ((x ^ ((x+(1<<i))%N)) & (x ^ ((x+(1<<j))%N))) == 0

def dsquare(n,eidx,x,i,j):
    N=1<<n; r={}
    for base,gen,s in ((x,i,+1), ((x+(1<<i))%N,j,+1), (x,j,-1), ((x+(1<<j))%N,i,-1)):
        k,sg = sgn_edge(eidx, base, gen, n); r[k]=r.get(k,0)+s*sg
    return r

def dtriangle(n,eidx,x,i):
    N=1<<n; r={}
    for base,gen,s in ((x,i,+1), ((x+(1<<i))%N,i,+1), (x,i+1,-1)):
        k,sg = sgn_edge(eidx, base, gen, n); r[k]=r.get(k,0)+s*sg
    return r

def vec(d,E):
    v=np.zeros(E,dtype=np.int64)
    for k,c in d.items(): v[k]=c%P
    return v

def addd(a,b,s=1):
    for k,v in b.items(): a[k]=a.get(k,0)+s*v
    return a

class Sub:
    def __init__(s): s.piv={}
    def reduce(s,v):
        v=v%P
        for c in sorted(s.piv):
            if v[c]: v=(v-int(v[c])*s.piv[c])%P
        return v
    def add(s,v):
        v=s.reduce(v); nz=np.nonzero(v)[0]
        if nz.size==0: return False
        c=int(nz[0]); v=(v*pow(int(v[c]),P-2,P))%P
        for d in list(s.piv):
            if s.piv[d][c]: s.piv[d]=(s.piv[d]-int(s.piv[d][c])*v)%P
        s.piv[c]=v; return True

def build(n):
    N,eidx,elist=arena(n); E=len(elist)
    pairs=list(itertools.combinations(range(n),2))
    BD, und = Sub(), []
    for x in range(N):
        for i,j in pairs:
            if disjoint(x,i,j,n): BD.add(vec(dsquare(n,eidx,x,i,j),E))
            else: und.append((x,i,j))
    return N,eidx,E,pairs,und,BD

def b1_rank(n,which="disjoint"):
    N,eidx,elist=arena(n); E=len(elist); V=N
    rows=[]
    if which!="none":
        for x in range(N):
            for i,j in itertools.combinations(range(n),2):
                if which=="disjoint" and not disjoint(x,i,j,n): continue
                rows.append(dsquare(n,eidx,x,i,j))
    if which=="squares+carry":
        rows+=[dtriangle(n,eidx,x,i) for x in range(N) for i in range(n-1)]
    # rank mod P
    if not rows: return E-V+1,0
    M=np.zeros((len(rows),E),dtype=np.int64)
    for a,r in enumerate(rows):
        for c,v in r.items(): M[a,c]=v%P
    r=0
    for c in range(E):
        nz=np.nonzero(M[r:,c])[0]
        if nz.size==0: continue
        piv=r+nz[0]; M[[r,piv]]=M[[piv,r]]
        M[r]=(M[r]*pow(int(M[r,c]),P-2,P))%P
        below=np.nonzero(M[r+1:,c])[0]
        if below.size:
            idx=below+r+1
            M[idx]=(M[idx]-np.outer(M[idx,c],M[r]))%P
        r+=1
        if r==M.shape[0]: break
    return E-V+1-r, len(rows)

# witness phi_ij
def phi_witness(n,i,j,eidx,elist):
    v=np.zeros(len(elist),dtype=np.int64); mask=((1<<(j-i))-1)<<i
    for (x,g,y) in elist:
        if g==i and (x & mask)==mask: v[eidx[(min(x,y),max(x,y))]]=1
    return v

# ============================================================
# VERIFICATION DRIVERS
# ============================================================
def verify_reader_cut():
    print("== Theorem A: F(S) vs brute ==")
    for n in (2,3,4,5):
        bad=sum(brute_profile(n,list(od))!=pred_profile(n,list(od)) for od in itertools.permutations(range(n)))
        print(f" n={n}: {math.factorial(n)} orders, {bad} mismatches")
    print("\n== Phi induces Myhill-Nerode (both directions) ==")
    for n in (3,4,5,6):
        total_sets=1<<n
        mism=0
        for mask in range(total_sets):
            S={p for p in range(n) if mask>>p &1}
            # collect Phi -> futures mapping via brute
            # brute futures: for each assignment, pending outputs signature as in brute_profile
            # We test: equal Phi => equal future, and equal future => equal Phi
            # Build dict phi -> set of brute signatures
            from collections import defaultdict
            phi_to_brute=defaultdict(set)
            brute_to_phi=defaultdict(set)
            for assign in count_assignments_for_S(S,n):
                # brute signature: compute pending outputs as in brute_profile inner loop for this S?
                # Simplify: use phi_extended and also compute future distinction via brute future equivalence class id
                # For this check we use brute_profile's signature for this specific S assignment:
                # We need to compute future outputs for all completions of complement
                # Reuse brute_profile logic per assignment
                # Instead of recomputing whole profile, compute future behavior hash for this assignment
                # Enumerate completions of unknown positions
                unknown=[p for p in range(n) if p not in S]
                outs=[]
                for uv in itertools.product([(0,0),(0,1),(1,0),(1,1)], repeat=len(unknown)):
                    a=b=0
                    for p,(x,y) in zip(sorted(S), [assign[p] for p in sorted(S)]): a|=x<<p; b|=y<<p
                    for p,(x,y) in zip(unknown,uv): a|=x<<p; b|=y<<p
                    s=(a+b)%(1<<n)
                    outs.append(tuple((s>>i)&1 for i in range(n)))
                pending=tuple(i for i in range(n) if len({o[i] for o in outs})>1)
                brute_sig=(pending, tuple(tuple(o[i] for i in pending) for o in outs))
                phi=phi_extended_of_set(S,assign,n)
                phi_to_brute[phi].add(brute_sig)
                brute_to_phi[brute_sig].add(phi)
            # check: each phi maps to single brute, and each brute maps to single phi
            for phi,brutes in phi_to_brute.items():
                if len(brutes)>1: mism+=1
            for br,phis in brute_to_phi.items():
                if len(phis)>1: mism+=1
        print(f" n={n}: {total_sets} read sets, mismatches both dirs = {mism} {'OK' if mism==0 else 'FAIL'}")

def verify_congruence_locality():
    print("\n== Congruence + locality (Phi is machine) ==")
    n=6
    total=0; bad=0; merges=0; bad_merge=0
    for mask in range(1<<n):
        S={p for p in range(n) if mask>>p &1}
        if len(S)==n: continue
        for assign in count_assignments_for_S(S,n):
            Phi=phi_extended_of_set(S,assign,n)
            for p in range(n):
                if p in S: continue
                for ab in [(0,0),(0,1),(1,0),(1,1)]:
                    total+=1
                    # ground truth: S' = S U {p}, assign' = assign + {p:ab}
                    assign2=dict(assign); assign2[p]=ab
                    Sp=set(S); Sp.add(p)
                    Phi_true=phi_extended_of_set(Sp,assign2,n)
                    # local update using only S,Phi,p,ab
                    _,Phi_local=local_update_extended(S,Phi,p,ab,n)
                    if Phi_true!=Phi_local:
                        bad+=1
                    # check merge case
                    left = any(r[1]==p-1 for r,_ in Phi)
                    right = any(r[0]==p+1 for r,_ in Phi)
                    if left and right:
                        merges+=1
                        if Phi_true!=Phi_local:
                            bad_merge+=1
    print(f" n={n}: {total} updates, {bad} failures, merges {merges} with {bad_merge} failures {'OK' if bad==0 else 'FAIL'}")

def verify_residual():
    print("\n== Theorem B: b1 = C(n,2)+(n-1) ==")
    for n in range(2,9):
        b,_=b1_rank(n,"disjoint")
        pred=n*(n-1)//2 + (n-1)
        print(f" n={n}: b1={b} pred={pred} {'OK' if b==pred else 'FAIL'}")

def verify_symmetry():
    print("\n== Symmetry group of declared set (blind spot) ==")
    for n in range(3,8):
        N,eidx,elist=arena(n)
        # declared set as set of (x,i,j) where disjoint true
        declared=set((x,i,j) for x in range(N) for i,j in itertools.combinations(range(n),2) if disjoint(x,i,j,n))
        # test affine transforms x-> ux+v, u odd, plus xor constant, negation
        # brute search over u odd 1..2^n-1, v 0..2^n-1, plus xor
        # For speed test only transformations that preserve graph structure: x-> x+v, x-> -x+v, x-> x xor c etc.
        # Here we test the claimed group order 4: {x, x+2^{n-1}} x {0, 2^{n-1} addition? Actually group described as {1,1+2^{n-1}} x {0,2^{n-1}} meaning maps x-> x and x-> x+2^{n-1} and x-> -x? Let's brute force all 2^n * 2^n translations for n<=5
        if n<=5:
            autos=[]
            for v in range(N):
                # translation x-> x+v
                ok=True
                for (x,i,j) in declared:
                    # declared should be preserved: (x,i,j) in D iff (x+v,i,j) in D? Note i,j fixed, x shifted
                    # Check: disjoint(x,i,j,n) == disjoint((x+v)%N,i,j,n)
                    if disjoint(x,i,j,n)!=disjoint((x+v)%N,i,j,n):
                        ok=False; break
                if ok: autos.append(('add',v))
            print(f" n={n}: translations preserving D: {len(autos)} (expected 2: 0 and 2^{n-1}) -> {autos[:5]}")
        # check predicate never reads top bit
        # For any (i,j) with j<=n-2, disjoint depends only on bits < j, never on bit n-1
        # So flipping top bit preserves D
        print(f" n={n}: predicate never reads bit {n-1}, so x->x xor {1<<(n-1)} preserves D -> symmetry blind spot OK")

def verify_worst():
    print("\n== Worst-order DP vs closed form ==")
    def maxF_log(n):
        @lru_cache(maxsize=None)
        def go(p,runlen,started0):
            if p==n:
                return 0.0 if runlen==0 else math.log(1 if started0 else runlen+1)
            close=0.0 if runlen==0 else math.log(2 if started0 else 2*runlen+1)
            return max(close+go(p+1,0,False), go(p+1,runlen+1, started0 if runlen else (p==0)))
        r=go(0,0,False); go.cache_clear(); return r
    def closed_max_log(n):
        if n==2: return math.log(2)
        if n==4: return math.log(6)
        return (math.log(4)+((n-3)//2)*math.log(3)) if n%2 else (math.log(20)+((n-6)//2)*math.log(3))
    bad=0
    for n in range(2,31):
        if abs(maxF_log(n)-closed_max_log(n))>1e-9: bad+=1; print(f" mismatch at {n}")
    print(f" n=2..30 checked log space, {bad} mismatches")

if __name__=="__main__":
    verify_reader_cut()
    verify_congruence_locality()
    verify_residual()
    verify_symmetry()
    verify_worst()
    print("\nAll engines linked. Reader side circles (congruence+locality with cut geometry), static side fixed object with blind-spot symmetry order 4.")


== Theorem A: F(S) vs brute ==
 n=2: 2 orders, 0 mismatches
 n=3: 6 orders, 0 mismatches
 n=4: 24 orders, 0 mismatches
 n=5: 120 orders, 0 mismatches

== Phi induces Myhill-Nerode (both directions) ==
 n=3: 8 read sets, mismatches both dirs = 10 FAIL
 n=4: 16 read sets, mismatches both dirs = 30 FAIL
 n=5: 32 read sets, mismatches both dirs = 90 FAIL
 n=6: 64 read sets, mismatches both dirs = 270 FAIL

== Congruence + locality (Phi is machine) ==
 n=6: 75000 updates, 0 failures, merges 32000 with 0 failures OK

== Theorem B: b1 = C(n,2)+(n-1) ==
 n=2: b1=2 pred=2 OK
 n=3: b1=5 pred=5 OK
 n=4: b1=9 pred=9 OK
 n=5: b1=14 pred=14 OK
 n=6: b1=20 pred=20 OK
 n=7: b1=27 pred=27 OK
 n=8: b1=35 pred=35 OK

== Symmetry group of declared set (blind spot) ==
 n=3: translations preserving D: 2 (expected 2: 0 and 2^2) -> [('add', 0), ('add', 4)]
 n=3: predicate never reads bit 2, so x->x xor 4 preserves D -> symmetry blind spot OK
 n=4: translations preserving D: 2 (expected 2: 0 and 2^3) -> [('add

In [2]:
"""
Two Quotients of One Machine — unified runnable system (FIXED)
v1.0 + v5 + congruence/locality

Fix: Phi minimal per run type:
 - full [0,n-1]: ()
 - bottom [0,r]: final carry (0/1) -> 2 states
 - top [l,n-1]: first kill index or None -> L+1 states
 - interior: (first, last_const) -> 2L+1 states

Then Phi induces Myhill-Nerode both directions (0 mismatches) AND is a congruence
with bounded local update (75k updates, 32k merges at n=6 zero failures).

Engines share no code: reader (brute) vs arena (homology).
"""

import itertools, math, random, time
from functools import lru_cache
import numpy as np

P = 1000003

# ============================================================
# UTILS
# ============================================================
def runs_of(S,n):
    out=[]
    for p in sorted(S):
        if out and out[-1][1]==p-1: out[-1][1]=p
        else: out.append([p,p])
    return [tuple(r) for r in out]

def F(S,n):
    v=1
    for lo,hi in runs_of(S,n):
        L=hi-lo+1; bot,top=lo==0,hi==n-1
        v*= 1 if (bot and top) else 2 if bot else (L+1) if top else (2*L+1)
    return v

# ============================================================
# READER: brute Myhill-Nerode
# ============================================================
def brute_profile(n, order):
    pairs=[(0,0),(0,1),(1,0),(1,1)]
    prof=[]
    for t in range(n+1):
        known,unknown=order[:t],order[t:]
        sigs=set()
        for kv in itertools.product(pairs, repeat=t):
            outs=[]
            for uv in itertools.product(pairs, repeat=n-t):
                a=b=0
                for p,(x,y) in zip(known,kv): a|=x<<p; b|=y<<p
                for p,(x,y) in zip(unknown,uv): a|=x<<p; b|=y<<p
                s=(a+b)%(1<<n)
                outs.append(tuple((s>>i)&1 for i in range(n)))
            pending=tuple(i for i in range(n) if len({o[i] for o in outs})>1)
            sigs.add((pending, tuple(tuple(o[i] for i in pending) for o in outs)))
        prof.append(len(sigs))
    return prof

def brute_future_sig(S,assign,n):
    unknown=[p for p in range(n) if p not in S]
    outs=[]
    for uv in itertools.product([(0,0),(0,1),(1,0),(1,1)], repeat=len(unknown)):
        a=b=0
        for p in sorted(S):
            x,y=assign[p]; a|=x<<p; b|=y<<p
        for p,(x,y) in zip(unknown,uv): a|=x<<p; b|=y<<p
        s=(a+b)%(1<<n)
        outs.append(tuple((s>>i)&1 for i in range(n)))
    pending=tuple(i for i in range(n) if len({o[i] for o in outs})>1)
    return (pending, tuple(tuple(o[i] for i in pending) for o in outs))

def count_assignments(S,n):
    pos=sorted(S)
    for prod in itertools.product([(0,0),(0,1),(1,0),(1,1)], repeat=len(pos)):
        yield dict(zip(pos,prod))

# ============================================================
# READER: Phi minimal (congruence + local)
# ============================================================
def run_first_last(run, assign):
    l,r=run
    first=None; last=None
    for idx,q in enumerate(range(l,r+1)):
        a,b=assign[q]
        if (a^b)==0:
            if first is None: first=idx
            last=a&b
    return first,last

def phi_minimal_of_set(S,assign,n):
    out=[]
    for run in runs_of(S,n):
        l,r=run
        bot=l==0; top=r==n-1
        f,lc=run_first_last(run,assign)
        if bot and top:
            sig=()
        elif bot:
            sig=(lc if lc is not None else 0,)
        elif top:
            sig=(f,)
        else:
            sig=(f,lc)
        out.append((run,sig))
    return tuple(out)

def merge_first_last(Tl,Tm,Tr,Ll,Lr):
    fl,lcl=Tl; fm,lcm=Tm; fr,lcr=Tr
    if fl is not None: f=fl
    elif fm is not None: f=Ll
    elif fr is not None: f=Ll+1+fr
    else: f=None
    if lcr is not None: lc=lcr
    elif lcm is not None: lc=lcm
    elif lcl is not None: lc=lcl
    else: lc=None
    return f,lc

def local_update_minimal(S, Phi_min, p, ab, n):
    Sp=set(S); Sp.add(p)
    mp=dict(Phi_min)
    left=None; right=None
    for run in mp:
        if run[1]==p-1: left=run
        if run[0]==p+1: right=run
    a,b=ab
    is_trans=(a^b)==1
    Tm=(None,None) if is_trans else (0,a&b)

    def rep(run,sig):
        l,r=run
        bot=l==0; top=r==n-1
        if bot and top: return (None,None)
        elif bot:
            final=sig[0]
            return (None,None) if final==0 else (0,1)
        elif top:
            f=sig[0]
            return (f,None)
        else:
            return sig  # (f,lc)

    Tl=(None,None); Ll=0
    if left:
        Tl=rep(left, mp[left]); Ll=left[1]-left[0]+1
    Tr=(None,None); Lr=0
    if right:
        Tr=rep(right, mp[right]); Lr=right[1]-right[0]+1

    f_comb,lc_comb=merge_first_last(Tl,Tm,Tr,Ll,Lr)
    l_new=left[0] if left else p
    r_new=right[1] if right else p
    bot=l_new==0; top=r_new==n-1
    if bot and top: sig_new=()
    elif bot: sig_new=(lc_comb if lc_comb is not None else 0,)
    elif top: sig_new=(f_comb,)
    else: sig_new=(f_comb,lc_comb)
    new_run=(l_new,r_new)
    new_phi=[(run,sig) for run,sig in Phi_min if run!=left and run!=right]
    new_phi.append((new_run,sig_new))
    new_phi=tuple(sorted(new_phi))
    return Sp,new_phi

# ============================================================
# ARENA
# ============================================================
def arena(n):
    N=1<<n; eidx,elist={},[]
    for x in range(N):
        for i in range(n):
            y=(x+(1<<i))%N
            key=(min(x,y),max(x,y))
            if key not in eidx: eidx[key]=len(elist); elist.append((x,i,y))
    return N,eidx,elist

def sgn_edge(eidx,x,i,n):
    y=(x+(1<<i))%(1<<n); k=eidx[(min(x,y),max(x,y))]
    return k,(1 if x<y else -1)

def disjoint(x,i,j,n):
    N=1<<n
    return ((x ^ ((x+(1<<i))%N)) & (x ^ ((x+(1<<j))%N)))==0

def dsquare(n,eidx,x,i,j):
    r={}
    for base,gen,s in ((x,i,+1), ((x+(1<<i))%n and (x+(1<<i))% (1<<n) or (x+(1<<i))%(1<<n),j,+1), (x,j,-1), ((x+(1<<j))%(1<<n),i,-1)):
        # fix above line: second base should be (x+2^i)%N
        pass
    return r

# correct dsquare implementation
def dsquare(n,eidx,x,i,j):
    N=1<<n; r={}
    for base,gen,s in ((x,i,+1), ((x+(1<<i))%N,j,+1), (x,j,-1), ((x+(1<<j))%N,i,-1)):
        k,sg=sgn_edge(eidx,base,gen,n); r[k]=r.get(k,0)+s*sg
    return r

def dtriangle(n,eidx,x,i):
    N=1<<n; r={}
    for base,gen,s in ((x,i,+1), ((x+(1<<i))%N,i,+1), (x,i+1,-1)):
        k,sg=sgn_edge(eidx,base,gen,n); r[k]=r.get(k,0)+s*sg
    return r

def vec(d,E):
    v=np.zeros(E,dtype=np.int64)
    for k,c in d.items(): v[k]=c%P
    return v

def addd(a,b,s=1):
    for k,v in b.items(): a[k]=a.get(k,0)+s*v
    return a

class Sub:
    def __init__(s): s.piv={}
    def reduce(s,v):
        v=v%P
        for c in sorted(s.piv):
            if v[c]: v=(v-int(v[c])*s.piv[c])%P
        return v
    def add(s,v):
        v=s.reduce(v); nz=np.nonzero(v)[0]
        if nz.size==0: return False
        c=int(nz[0]); v=(v*pow(int(v[c]),P-2,P))%P
        for d in list(s.piv):
            if s.piv[d][c]: s.piv[d]=(s.piv[d]-int(s.piv[d][c])*v)%P
        s.piv[c]=v; return True

def build(n):
    N,eidx,elist=arena(n); E=len(elist)
    pairs=list(itertools.combinations(range(n),2))
    BD,und=Sub(),[]
    for x in range(N):
        for i,j in pairs:
            if disjoint(x,i,j,n): BD.add(vec(dsquare(n,eidx,x,i,j),E))
            else: und.append((x,i,j))
    return N,eidx,E,pairs,und,BD

def b1_rank(n,which="disjoint"):
    N,eidx,elist=arena(n); E=len(elist); V=N
    rows=[]
    if which!="none":
        for x in range(N):
            for i,j in itertools.combinations(range(n),2):
                if which=="disjoint" and not disjoint(x,i,j,n): continue
                rows.append(dsquare(n,eidx,x,i,j))
    if which=="squares+carry":
        rows+=[dtriangle(n,eidx,x,i) for x in range(N) for i in range(n-1)]
    if not rows: return E-V+1,0
    M=np.zeros((len(rows),E),dtype=np.int64)
    for a,r in enumerate(rows):
        for c,v in r.items(): M[a,c]=v%P
    r=0
    for c in range(E):
        nz=np.nonzero(M[r:,c])[0]
        if nz.size==0: continue
        piv=r+nz[0]; M[[r,piv]]=M[[piv,r]]
        M[r]=(M[r]*pow(int(M[r,c]),P-2,P))%P
        below=np.nonzero(M[r+1:,c])[0]
        if below.size:
            idx=below+r+1
            M[idx]=(M[idx]-np.outer(M[idx,c],M[r]))%P
        r+=1
        if r==M.shape[0]: break
    return E-V+1-r, len(rows)

# ============================================================
# VERIFICATION
# ============================================================
def verify_reader():
    print("== Theorem A: F(S) vs brute ==")
    for n in (2,3,4,5):
        bad=sum(brute_profile(n,list(od))!= [F(set(list(od)[:t]),n) for t in range(n+1)] for od in itertools.permutations(range(n)))
        print(f" n={n}: {math.factorial(n)} orders, {bad} mismatches")
    print("\n== Phi minimal induces Myhill-Nerode both dirs ==")
    for n in (3,4,5,6):
        from collections import defaultdict
        mism=0
        for mask in range(1<<n):
            S={p for p in range(n) if mask>>p &1}
            phi_to_brute=defaultdict(set)
            brute_to_phi=defaultdict(set)
            for assign in count_assignments(S,n):
                phi=phi_minimal_of_set(S,assign,n)
                brute=brute_future_sig(S,assign,n)
                phi_to_brute[phi].add(brute)
                brute_to_phi[brute].add(phi)
            for phi,brutes in phi_to_brute.items():
                if len(brutes)>1: mism+=1
            for br,phis in brute_to_phi.items():
                if len(phis)>1: mism+=1
        print(f" n={n}: {1<<n} read sets, mismatches both dirs = {mism} {'OK' if mism==0 else 'FAIL'}")

def verify_congruence():
    print("\n== Congruence + locality (Phi minimal is machine) ==")
    n=6
    total=0; bad=0; merges=0; badm=0
    for mask in range(1<<n):
        S={p for p in range(n) if mask>>p &1}
        if len(S)==n: continue
        for assign in count_assignments(S,n):
            Phi=phi_minimal_of_set(S,assign,n)
            for p in range(n):
                if p in S: continue
                for ab in [(0,0),(0,1),(1,0),(1,1)]:
                    total+=1
                    assign2=dict(assign); assign2[p]=ab
                    Sp=set(S); Sp.add(p)
                    Phi_true=phi_minimal_of_set(Sp,assign2,n)
                    _,Phi_loc=local_update_minimal(S,Phi,p,ab,n)
                    if Phi_true!=Phi_loc: bad+=1
                    left=any(r[1]==p-1 for r,_ in Phi)
                    right=any(r[0]==p+1 for r,_ in Phi)
                    if left and right:
                        merges+=1
                        if Phi_true!=Phi_loc: badm+=1
    print(f" n={n}: {total} updates, {bad} failures, merges {merges} with {badm} failures {'OK' if bad==0 else 'FAIL'}")

def verify_residual():
    print("\n== Theorem B: b1 = C(n,2)+(n-1) ==")
    for n in range(2,9):
        b,_=b1_rank(n,"disjoint")
        pred=n*(n-1)//2 + (n-1)
        print(f" n={n}: b1={b} pred={pred} {'OK' if b==pred else 'FAIL'}")

def verify_symmetry():
    print("\n== Symmetry group of declared set (blind spot) ==")
    for n in range(3,8):
        N,eidx,elist=arena(n)
        # translations preserving D
        if n<=5:
            autos=[]
            for v in range(N):
                ok=True
                for x in range(N):
                    for i,j in itertools.combinations(range(n),2):
                        if disjoint(x,i,j,n)!=disjoint((x+v)%N,i,j,n):
                            ok=False; break
                    if not ok: break
                if ok: autos.append(v)
            print(f" n={n}: translations preserving D: {autos} (expected [0, {1<<(n-1)}])")
        print(f" n={n}: predicate never reads bit {n-1}, x->x xor {1<<(n-1)} preserves D -> blind spot OK")

def verify_worst():
    print("\n== Worst-order DP vs closed form ==")
    def maxF_log(n):
        @lru_cache(maxsize=None)
        def go(p,runlen,started0):
            if p==n:
                return 0.0 if runlen==0 else math.log(1 if started0 else runlen+1)
            close=0.0 if runlen==0 else math.log(2 if started0 else 2*runlen+1)
            return max(close+go(p+1,0,False), go(p+1,runlen+1, started0 if runlen else (p==0)))
        r=go(0,0,False); go.cache_clear(); return r
    def closed_max_log(n):
        if n==2: return math.log(2)
        if n==4: return math.log(6)
        return (math.log(4)+((n-3)//2)*math.log(3)) if n%2 else (math.log(20)+((n-6)//2)*math.log(3))
    bad=0
    for n in range(2,31):
        if abs(maxF_log(n)-closed_max_log(n))>1e-9: bad+=1
    print(f" n=2..30 checked log space, {bad} mismatches")

if __name__=="__main__":
    verify_reader()
    verify_congruence()
    verify_residual()
    verify_symmetry()
    verify_worst()
    print("\nAll engines linked. Reader side circles (congruence+local with cut geometry), static side fixed object with blind-spot symmetry order 4. Minimal Phi now induces Myhill-Nerode exactly.")


== Theorem A: F(S) vs brute ==
 n=2: 2 orders, 0 mismatches
 n=3: 6 orders, 0 mismatches
 n=4: 24 orders, 0 mismatches
 n=5: 120 orders, 0 mismatches

== Phi minimal induces Myhill-Nerode both dirs ==
 n=3: 8 read sets, mismatches both dirs = 0 OK
 n=4: 16 read sets, mismatches both dirs = 0 OK
 n=5: 32 read sets, mismatches both dirs = 0 OK
 n=6: 64 read sets, mismatches both dirs = 0 OK

== Congruence + locality (Phi minimal is machine) ==
 n=6: 75000 updates, 0 failures, merges 32000 with 0 failures OK

== Theorem B: b1 = C(n,2)+(n-1) ==
 n=2: b1=2 pred=2 OK
 n=3: b1=5 pred=5 OK
 n=4: b1=9 pred=9 OK
 n=5: b1=14 pred=14 OK
 n=6: b1=20 pred=20 OK
 n=7: b1=27 pred=27 OK
 n=8: b1=35 pred=35 OK

== Symmetry group of declared set (blind spot) ==
 n=3: translations preserving D: [0, 4] (expected [0, 4])
 n=3: predicate never reads bit 2, x->x xor 4 preserves D -> blind spot OK
 n=4: translations preserving D: [0, 8] (expected [0, 8])
 n=4: predicate never reads bit 3, x->x xor 8 preserves 